In [1]:
!pip -q install sentence-transformers faiss-cpu transformers sentencepiece

In [1]:
import pandas as pd
import numpy as np
import faiss
import pickle
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display

In [2]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA available: False
Running on CPU


In [3]:
#Load the chunks

chunks_path = "./rmit_assessment_support_chunks.csv"

chunks_df = pd.read_csv(chunks_path)

print("Chunks loaded:", len(chunks_df))
print("Columns:")
print(chunks_df.columns.tolist())

display(chunks_df.head())

Chunks loaded: 71
Columns:
['chunk_id', 'source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date', 'document_version', 'chunk_number', 'total_chunks', 'chunk_word_count']


,chunk_id,source_id,category,title,section,content,source_url,source_type,collection_date,effective_date,document_version,chunk_number,total_chunks,chunk_word_count
0,EXT-01_0,EXT-01,Extension,Extensions,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1,90
1,EXT-01_1,EXT-01,Extension,Extensions,Assessments eligible for an extension,You can apply for an extension for assessments...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1,50
2,EXT-01_2,EXT-01,Extension,Extensions,How to apply,You must apply at least one working day before...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1,213
3,EXT-01_3,EXT-01,Extension,Extensions,False documents and misleading information,"Creating, submitting or using fraudulent docum...",https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1,90
4,SC-01_4,SC-01,Special Consideration,Special consideration,Special consideration,If unexpected circumstances beyond your contro...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1,46


In [5]:
# Load the FAISS index
index_path = "./rmit_assessment_support.faiss"

index = faiss.read_index(index_path)

print("FAISS index loaded successfully.")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", index.d)

FAISS index loaded successfully.
Number of vectors: 71
Vector dimension: 384


In [6]:
#Load the metadata
metadata_path = "./rmit_assessment_support_metadata.pkl"

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("Metadata loaded successfully.")
print("Number of metadata records:", len(metadata))

Metadata loaded successfully.
Number of metadata records: 71


In [7]:
#Load the embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


Documents
   ↓
all-MiniLM-L6-v2
   ↓
384-dimensional vectors


Student question
   ↓
all-MiniLM-L6-v2
   ↓
384-dimensional vector

In [8]:
#Recreate the retrieval function
def retrieve_documents(query, k=5):

    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Normalize the query embedding
    faiss.normalize_L2(query_embedding)

    # Search the FAISS index
    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        result = metadata[idx].copy()
        result["similarity_score"] = float(score)

        results.append(result)

    return pd.DataFrame(results)

In [9]:
#Test retrieval again
query = "How many days can an assessment extension be approved for?"

results = retrieve_documents(
    query,
    k=5
)

display(
    results[
        [
            "similarity_score",
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

,similarity_score,source_id,category,section,content
0,0.835225,EXT-01,Extension,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...
1,0.789507,POL-01,Policy,Extensions,(34) Extensions are available for unforeseen c...
2,0.742802,EAA-02,Equitable Assessment Arrangements,How do I ask for an extension?,If you have the Equitable Assessment Arrangeme...
3,0.706117,EAA-02,Equitable Assessment Arrangements,How many days can I request for an extension (...,If you have the Equitable Assessment Arrangeme...
4,0.691605,EAA-02,Equitable Assessment Arrangements,Can I request an extension for a group assessm...,Extensions in your ELP only apply to individua...


In [10]:
#Load the language model
llm_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    llm_name
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    llm_name
)

print("LLM loaded successfully.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully.


What is FLAN-T5?

FLAN-T5 is an instruction-tuned language model

                Student question
                       │
              ┌────────┴────────┐
              │                 │
              ▼                 │
      Embedding model           │
    all-MiniLM-L6-v2            │
              │                 │
              ▼                 │
           FAISS                │
              │                 │
              ▼                 │
      Relevant RMIT chunks      │
              │                 │
              └────────┬────────┘
                       ▼
                Context + Question
                       │
                       ▼
                  FLAN-T5
                       │
                       ▼
                 Final answer

In [11]:
#Build the RAG answer function
def generate_rag_answer(query, k=5):

    # Step 1: Retrieve relevant RMIT chunks
    results = retrieve_documents(query, k=k)

    # Step 2: Combine retrieved chunks into context
    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"Source: {row['source_id']}\n"
            f"Section: {row['section']}\n"
            f"Content: {row['content']}"
        )

    context = "\n\n".join(context_parts)

    # Step 3: Create a grounded prompt
    prompt = f"""
You are an RMIT University student assessment support assistant.

Answer the student's question using ONLY the information provided
in the RMIT context below.

Do not invent policies, deadlines, requirements, or procedures.

If the context does not contain enough information to answer the
question, say that the available RMIT information does not provide
enough information and recommend checking the relevant RMIT source.

RMIT CONTEXT:
{context}

STUDENT QUESTION:
{query}

ANSWER:
"""

    # Step 4: Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Step 5: Generate the answer
    with torch.no_grad():

        output = llm.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            do_sample=False
        )

    # Step 6: Convert generated tokens back to text
    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return {
        "question": query,
        "answer": answer,
        "retrieved_documents": results
    }

In [12]:
# Test your first complete RAG answer

query = "How many days can an assessment extension be approved for?"

rag_result = generate_rag_answer(
    query,
    k=5
)

print("QUESTION:")
print(rag_result["question"])

print("\nANSWER:")
print(rag_result["answer"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION:
How many days can an assessment extension be approved for?

ANSWER:
You may apply for an extension to the due date of up to seven calendar days.


In [13]:
display(
    rag_result["retrieved_documents"][
        [
            "similarity_score",
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

,similarity_score,source_id,category,section,content
0,0.835225,EXT-01,Extension,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...
1,0.789507,POL-01,Policy,Extensions,(34) Extensions are available for unforeseen c...
2,0.742802,EAA-02,Equitable Assessment Arrangements,How do I ask for an extension?,If you have the Equitable Assessment Arrangeme...
3,0.706117,EAA-02,Equitable Assessment Arrangements,How many days can I request for an extension (...,If you have the Equitable Assessment Arrangeme...
4,0.691605,EAA-02,Equitable Assessment Arrangements,Can I request an extension for a group assessm...,Extensions in your ELP only apply to individua...


Student question

      ↓
all-MiniLM-L6-v2

      ↓
Query embedding

      ↓
FAISS

      ↓
Top 5 RMIT chunks

      ↓
Grounded prompt

      ↓
FLAN-T5

      ↓
Answer